# 🔬 Lab Module 25: Agent Orchestration at Scale

**Mục tiêu:** Simulate distributed LoanBot fleet — message queue, auto-scaling, Saga pattern, distributed tracing.

| Section | Topic |
|---------|-------|
| 1 | Stateless Worker + Redis State Pattern (simulated) |
| 2 | Message Queue với At-Least-Once + Idempotency |
| 3 | Saga Pattern — Forward + Compensating Transactions |
| 4 | Auto-Scaler — Little's Law + Queue-Depth Scaling |
| 5 | Distributed Tracing — Correlation ID + Span Collection |
| 6 | SLO Monitor — p99 Latency + Error Rate Alerts |
| 7 | Full Fleet Simulation (TODO + Solution) |

In [ ]:
import uuid
import time
import random
import json
import statistics
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable
from enum import Enum
from collections import deque

# Simulated Redis (in-memory for lab)
class SimulatedRedis:
    def __init__(self):
        self._store: Dict[str, str] = {}
        self._counters: Dict[str, int] = {}
        self._stream: List[dict] = []
        self._acked: set = set()
        self._pending: Dict[str, str] = {}  # msg_id -> consumer

    def set(self, key, value, nx=False, ex=None):
        if nx and key in self._store: return False
        self._store[key] = value
        return True

    def get(self, key): return self._store.get(key)
    def delete(self, key): self._store.pop(key, None)

    def incr(self, key):
        self._counters[key] = self._counters.get(key, 0) + 1
        return self._counters[key]

    def xadd(self, stream, data):
        msg_id = f'{len(self._stream):06d}-0'
        self._stream.append({'id': msg_id, 'data': data})
        return msg_id

    def xreadgroup(self, consumer, count=10):
        results = []
        for msg in self._stream:
            if msg['id'] not in self._acked and msg['id'] not in self._pending:
                self._pending[msg['id']] = consumer
                results.append((msg['id'], msg['data']))
                if len(results) >= count: break
        return results

    def xack(self, msg_id):
        self._acked.add(msg_id)
        self._pending.pop(msg_id, None)

    def queue_depth(self):
        return sum(1 for m in self._stream if m['id'] not in self._acked)

redis_sim = SimulatedRedis()
print('✅ SimulatedRedis initialized')

## Section 1: Stateless Worker Pattern

In [ ]:
@dataclass
class LoanApplication:
    app_id: str
    credit_score: int
    dti_ratio: float
    loan_amount: int
    employment_months: int

class StatelessLoanBotWorker:
    def __init__(self, worker_id: str, redis: SimulatedRedis, seed: int = 42):
        self.worker_id = worker_id
        self.redis = redis
        self.rng = random.Random(seed + hash(worker_id) % 100)

    def process(self, app: LoanApplication) -> dict:
        # Deduplication via distributed lock
        lock_key = f'lock:app:{app.app_id}'
        acquired = self.redis.set(lock_key, self.worker_id, nx=True)
        if not acquired:
            return {'status': 'duplicate_skipped', 'app_id': app.app_id, 'worker': self.worker_id}

        try:
            # Check result cache (idempotency)
            cached_key = f'result:{app.app_id}'
            cached = self.redis.get(cached_key)
            if cached:
                return {'status': 'cached', 'app_id': app.app_id, 'result': json.loads(cached)}

            # Stateless decision (no local state)
            decision = self._decide(app)

            # Save to 'DB' via Redis (simulated)
            self.redis.set(cached_key, json.dumps(decision))
            counter = self.redis.incr('counter:decisions:total')

            return {'status': 'processed', 'app_id': app.app_id, 'decision': decision, 'total_count': counter, 'worker': self.worker_id}
        finally:
            self.redis.delete(lock_key)

    def _decide(self, app: LoanApplication) -> str:
        if app.credit_score >= 680 and app.dti_ratio <= 0.42: return 'APPROVE'
        elif app.credit_score >= 580 or app.dti_ratio <= 0.45: return 'REVIEW'
        return 'REJECT'

# Test: 2 workers, 1 app — demonstrate deduplication
worker_a = StatelessLoanBotWorker('worker-A', redis_sim, seed=1)
worker_b = StatelessLoanBotWorker('worker-B', redis_sim, seed=2)
app1 = LoanApplication('FC-2025-001', 720, 0.32, 300_000_000, 24)

result_a = worker_a.process(app1)
result_b = worker_b.process(app1)  # Same app — should be cached

print(f'Worker A: {result_a["status"]} — decision: {result_a.get("decision", result_a.get("result", "N/A"))}')
print(f'Worker B: {result_b["status"]} — (should be cached, not duplicate_skipped)')
print(f'Total decisions counter: {redis_sim._counters.get("counter:decisions:total", 0)}')

## Section 2: Message Queue — At-Least-Once + DLQ

In [ ]:
@dataclass
class QueueMessage:
    msg_id: str
    payload: dict
    retry_count: int = 0
    enqueued_at: float = field(default_factory=time.time)

class LoanBotTaskQueue:
    MAX_RETRIES = 3

    def __init__(self):
        self._queue: deque = deque()
        self._dlq: List[QueueMessage] = []
        self._acked: set = set()

    def enqueue(self, app: LoanApplication) -> str:
        msg_id = f'msg-{uuid.uuid4().hex[:8]}'
        self._queue.append(QueueMessage(msg_id=msg_id, payload={
            'app_id': app.app_id, 'credit_score': app.credit_score,
            'dti_ratio': app.dti_ratio, 'loan_amount': app.loan_amount
        }))
        return msg_id

    def consume(self, batch_size: int = 5) -> List[QueueMessage]:
        batch = []
        for _ in range(min(batch_size, len(self._queue))):
            if self._queue: batch.append(self._queue.popleft())
        return batch

    def ack(self, msg_id: str): self._acked.add(msg_id)

    def nack(self, msg: QueueMessage, error: str):
        msg.retry_count += 1
        if msg.retry_count >= self.MAX_RETRIES:
            self._dlq.append(msg)
            print(f'  ❌ → DLQ: {msg.msg_id} ({error}) after {msg.retry_count} retries')
        else:
            self._queue.append(msg)  # Re-enqueue
            print(f'  ↩️ Retry {msg.retry_count}: {msg.msg_id}')

    @property
    def depth(self): return len(self._queue)

    def dlq_report(self):
        return [{'msg_id': m.msg_id, 'retry_count': m.retry_count, 'app_id': m.payload['app_id']} for m in self._dlq]

queue = LoanBotTaskQueue()
apps = [
    LoanApplication(f'FC-2025-{i:03d}', 500 + i*30, 0.30 + i*0.02, 200_000_000, 12)
    for i in range(10)
]
for app in apps: queue.enqueue(app)
print(f'Enqueued {queue.depth} tasks\n')

# Simulate processing with occasional failures
rng = random.Random(42)
worker = StatelessLoanBotWorker('worker-1', SimulatedRedis(), seed=1)

print('=== Processing with simulated failures ===')
round_num = 0
while queue.depth > 0 or round_num == 0:
    batch = queue.consume(batch_size=3)
    if not batch: break
    round_num += 1
    print(f'\nRound {round_num}: processing {len(batch)} tasks (queue depth after: {queue.depth})')
    for msg in batch:
        app = LoanApplication(msg.payload['app_id'], msg.payload['credit_score'], msg.payload['dti_ratio'], msg.payload['loan_amount'], 12)
        # Simulate 20% failure rate
        if rng.random() < 0.20:
            queue.nack(msg, 'CIC API timeout')
        else:
            result = worker.process(app)
            queue.ack(msg.msg_id)
            print(f'  ✅ {msg.msg_id} → {result.get("decision", result["status"])}')

print(f'\n📬 DLQ items: {len(queue._dlq)}')
for item in queue.dlq_report():
    print(f'  {item}')

## Section 3: Saga Pattern

In [ ]:
class SagaStatus(Enum):
    PENDING = 'pending'
    SUCCESS = 'success'
    COMPENSATED = 'compensated'
    FAILED = 'failed'

@dataclass
class SagaStep:
    name: str
    execute: Callable
    compensate: Callable
    status: SagaStatus = SagaStatus.PENDING
    result: Optional[dict] = None

class LoanApprovalSaga:
    def __init__(self, app: LoanApplication, failure_at: str = None):
        self.app = app
        self.failure_at = failure_at
        self.state = {}
        self.steps = self._build_steps()

    def _build_steps(self) -> List[SagaStep]:
        return [
            SagaStep('S1_reserve_slot',
                execute=lambda: self._s1_reserve(),
                compensate=lambda: print(f'  [C1] Release processing slot for {self.app.app_id}')),
            SagaStep('S2_cic_query',
                execute=lambda: self._s2_cic(),
                compensate=lambda: print(f'  [C2] Invalidate CIC cache for {self.app.app_id}')),
            SagaStep('S3_loan_decision',
                execute=lambda: self._s3_decision(),
                compensate=lambda: print(f'  [C3] Reverse decision for {self.app.app_id} (if not yet notified)')),
            SagaStep('S4_notify_customer',
                execute=lambda: self._s4_notify(),
                compensate=lambda: print(f'  [C4] Notification already sent — cannot unsend')),
            SagaStep('S5_record_audit',
                execute=lambda: self._s5_audit(),
                compensate=lambda: print(f'  [C5] Audit log: record saga compensation')),
        ]

    def _s1_reserve(self): print(f'  [S1] Slot reserved for {self.app.app_id}'); return {'slot': 'reserved'}
    def _s2_cic(self): print(f'  [S2] CIC queried: score={self.app.credit_score}'); return {'cic_ok': True}
    def _s3_decision(self):
        if self.failure_at == 'S3': raise Exception('LoanBot model timeout')
        d = 'APPROVE' if self.app.credit_score >= 680 and self.app.dti_ratio <= 0.42 else 'REVIEW'
        print(f'  [S3] Decision: {d}')
        return {'decision': d}
    def _s4_notify(self):
        if self.failure_at == 'S4': raise Exception('SMS gateway timeout')
        print(f'  [S4] SMS sent to customer')
        return {'notified': True}
    def _s5_audit(self):
        if self.failure_at == 'S5': raise Exception('DB timeout')
        print(f'  [S5] Audit record written')
        return {'audit': True}

    def execute(self) -> SagaStatus:
        executed = []
        for step in self.steps:
            try:
                step.result = step.execute()
                step.status = SagaStatus.SUCCESS
                executed.append(step)
            except Exception as e:
                print(f'  ❌ {step.name} FAILED: {e}')
                step.status = SagaStatus.FAILED
                # Compensate in reverse order
                print(f'  → Compensating {len(executed)} completed steps:')
                for s in reversed(executed):
                    if s.name not in ['S4_notify_customer']:  # Cannot unsend notification
                        s.compensate()
                        s.status = SagaStatus.COMPENSATED
                return SagaStatus.COMPENSATED
        return SagaStatus.SUCCESS

# Test scenarios
app_good = LoanApplication('FC-2025-010', 720, 0.32, 500_000_000, 24)
app_bad  = LoanApplication('FC-2025-011', 580, 0.41, 300_000_000, 8)

print('=== Scenario 1: Happy Path ===')
result = LoanApprovalSaga(app_good).execute()
print(f'Final: {result.value}\n')

print('=== Scenario 2: S3 Failure (before notification) ===')
result = LoanApprovalSaga(app_bad, failure_at='S3').execute()
print(f'Final: {result.value}\n')

print('=== Scenario 3: S5 Failure (after notification already sent) ===')
result = LoanApprovalSaga(app_good, failure_at='S5').execute()
print(f'Final: {result.value}')

## Section 4: Auto-Scaler — Little's Law

In [ ]:
@dataclass
class AutoScaleConfig:
    min_workers: int = 2
    max_workers: int = 20
    scale_up_threshold: int = 50
    scale_down_threshold: int = 5
    cooldown_seconds: int = 60
    workers_per_scale_up: int = 2

def compute_required_workers(arrival_rate: float, avg_latency_s: float, buffer_pct: float = 0.20) -> int:
    """Little's Law: L = λ × W → workers = ceil(λ × W) × (1 + buffer)"""
    concurrent = arrival_rate * avg_latency_s
    return max(1, int(concurrent * (1 + buffer_pct)) + 1)

# Simulate traffic pattern over 8 hours
print('=== Little\'s Law — Optimal Worker Count per Hour ===')
print(f'{"Hour":>6} | {"Req/hr":>8} | {"λ (req/s)":>10} | {"L=λ×W":>8} | {"Workers needed":>15}')
print('-' * 60)

hourly_requests = [500, 2000, 8000, 12000, 10000, 7000, 4000, 1000]  # realistic pattern
avg_latency = 2.5  # seconds

for hour, reqs in enumerate(hourly_requests, start=9):
    lam = reqs / 3600
    L = lam * avg_latency
    workers = compute_required_workers(lam, avg_latency)
    bar = '█' * min(workers, 20)
    print(f'{hour:02d}:00  | {reqs:>8,} | {lam:>10.3f} | {L:>8.2f} | {workers:>5} {bar}')

print(f'\nKey insight: Peak hour 12:00 needs {compute_required_workers(12000/3600, 2.5)} workers')
print(f'Off-peak 09:00 needs only {compute_required_workers(500/3600, 2.5)} workers')
print(f'Auto-scaling saves ~{(compute_required_workers(12000/3600, 2.5) - compute_required_workers(500/3600, 2.5))} × cost of 1 worker during off-peak hours')

## Section 5: Distributed Tracing

In [ ]:
from contextlib import contextmanager

@dataclass
class TraceSpan:
    name: str
    correlation_id: str
    trace_id: str
    worker_id: str
    start_time: float = field(default_factory=time.time)
    end_time: Optional[float] = None
    status: str = 'in_progress'
    metadata: dict = field(default_factory=dict)

    def finish(self, status='ok', **meta):
        self.end_time = time.time()
        self.status = status
        self.metadata.update(meta)

    @property
    def duration_ms(self): return round((self.end_time - self.start_time) * 1000, 1) if self.end_time else -1

class LoanBotTracer:
    def __init__(self, worker_id: str):
        self.worker_id = worker_id
        self.spans: List[TraceSpan] = []

    @contextmanager
    def span(self, name: str, correlation_id: str, trace_id: str = None):
        s = TraceSpan(name, correlation_id, trace_id or str(uuid.uuid4())[:8], self.worker_id)
        self.spans.append(s)
        try:
            yield s
            s.finish(status='ok')
        except Exception as e:
            s.finish(status='error', error=str(e))
            raise

    def export(self) -> List[dict]:
        return [{'span': s.name, 'corr_id': s.correlation_id, 'duration_ms': s.duration_ms,
                 'status': s.status, 'worker': s.worker_id, **s.metadata} for s in self.spans]

# Simulate processing with tracing
def simulate_loan_processing(app_id: str, tracer: LoanBotTracer, simulate_slow_cic=False):
    corr_id = f'corr-{app_id}'
    trace_id = str(uuid.uuid4())[:8]

    with tracer.span('gateway_receive', corr_id, trace_id) as s:
        time.sleep(0.001)  # 1ms
        s.metadata['app_id'] = app_id

    with tracer.span('queue_wait', corr_id, trace_id) as s:
        time.sleep(0.005)  # 5ms queue wait

    with tracer.span('cic_api_call', corr_id, trace_id) as s:
        latency = 0.100 if not simulate_slow_cic else 0.800
        time.sleep(latency)
        if simulate_slow_cic and latency > 0.5:
            s.metadata['warning'] = 'slow_cic'

    with tracer.span('loanbot_decision', corr_id, trace_id) as s:
        time.sleep(0.020)  # 20ms LLM call (cached)
        s.metadata['decision'] = 'APPROVE'

    with tracer.span('db_write', corr_id, trace_id) as s:
        time.sleep(0.005)  # 5ms DB write

tracer = LoanBotTracer('worker-3')
simulate_loan_processing('FC-2025-001', tracer, simulate_slow_cic=False)
simulate_loan_processing('FC-2025-002', tracer, simulate_slow_cic=True)

print('=== Distributed Trace Export ===')
total_by_corr = {}
for span in tracer.export():
    cid = span['corr_id']
    total_by_corr.setdefault(cid, 0)
    total_by_corr[cid] += span['duration_ms']
    warn = ' ⚠️ SLOW' if span.get('warning') else ''
    print(f"  {cid} | {span['span']:20s} | {span['duration_ms']:6.1f}ms | {span['status']}{warn}")

print(f'\nEnd-to-end latency:')
for corr, total in total_by_corr.items():
    print(f'  {corr}: {total:.1f}ms total {"🚨 SLO breach!" if total > 8000 else "✅"}')

## Section 6: SLO Monitor

In [ ]:
@dataclass
class SLOTargets:
    p99_latency_ms: float = 8000
    error_rate_pct: float = 0.5
    queue_depth_max: int = 200
    throughput_min: int = 500  # per hour

def compute_slo_report(latencies_ms: List[float], errors: int, queue_depth: int, slo: SLOTargets) -> dict:
    sorted_lat = sorted(latencies_ms)
    p50 = sorted_lat[int(len(sorted_lat) * 0.50)] if sorted_lat else 0
    p99 = sorted_lat[int(len(sorted_lat) * 0.99)] if len(sorted_lat) >= 100 else sorted_lat[-1] if sorted_lat else 0
    error_rate = errors / len(latencies_ms) * 100 if latencies_ms else 0
    throughput = len(latencies_ms)  # per interval

    return {
        'p50_ms': round(p50, 1), 'p99_ms': round(p99, 1),
        'error_rate_pct': round(error_rate, 2),
        'queue_depth': queue_depth,
        'throughput': throughput,
        'slo_p99_ok': p99 <= slo.p99_latency_ms,
        'slo_errors_ok': error_rate <= slo.error_rate_pct,
        'slo_queue_ok': queue_depth <= slo.queue_depth_max,
        'slo_throughput_ok': throughput >= slo.throughput_min,
    }

# Simulate different load scenarios
rng = random.Random(42)
slo = SLOTargets()

scenarios = [
    ('Normal load', [rng.gauss(2500, 400) for _ in range(500)], 2, 30),
    ('Peak load', [rng.gauss(6000, 1500) for _ in range(500)], 8, 180),
    ('Degraded CIC', [rng.gauss(9000, 2000) for _ in range(500)], 25, 350),
]

print('=== SLO Report (SLO: p99<8000ms, errors<0.5%, queue<200, throughput>500) ===')
for name, latencies, errors, depth in scenarios:
    report = compute_slo_report(latencies, errors, depth, slo)
    all_ok = all([report['slo_p99_ok'], report['slo_errors_ok'], report['slo_queue_ok'], report['slo_throughput_ok']])
    status = '✅ ALL SLOs MET' if all_ok else '🚨 SLO BREACH'
    print(f'\n[{name}] {status}')
    print(f"  p50={report['p50_ms']:.0f}ms | p99={report['p99_ms']:.0f}ms {'✅' if report['slo_p99_ok'] else '❌'}")
    print(f"  errors={report['error_rate_pct']:.2f}% {'✅' if report['slo_errors_ok'] else '❌'} | queue={report['queue_depth']} {'✅' if report['slo_queue_ok'] else '❌'}")

## Section 7: Full Fleet Simulation

**TODO:** Implement `LoanBotFleet.run_simulation()` kết hợp tất cả components.
- Bước 1: Enqueue 100 loan applications
- Bước 2: 5 stateless workers consume từ queue (parallel simulation)
- Bước 3: Auto-scaler monitor queue depth và scale lên/xuống
- Bước 4: Track traces và compute SLO metrics
- Bước 5: Report: throughput, p99 latency, worker utilization, DLQ size

In [ ]:
class LoanBotFleet:
    def __init__(self, n_workers: int = 5):
        self.n_workers = n_workers
        self.queue = LoanBotTaskQueue()
        self.redis = SimulatedRedis()
        self.tracer = LoanBotTracer('fleet')

    def run_simulation(self, applications: List[LoanApplication]) -> dict:
        """TODO: Implement fleet simulation."""
        raise NotImplementedError('Implement this method!')

class LoanBotFleetSolution(LoanBotFleet):
    def run_simulation(self, applications: List[LoanApplication]) -> dict:
        rng = random.Random(42)
        for app in applications: self.queue.enqueue(app)
        print(f'📬 Enqueued {len(applications)} applications\n')

        workers = [StatelessLoanBotWorker(f'worker-{i}', self.redis, seed=i) for i in range(self.n_workers)]
        latencies, errors = [], 0
        round_n = 0

        while self.queue.depth > 0:
            round_n += 1
            # Each worker consumes up to 2 tasks
            for i, worker in enumerate(workers):
                batch = self.queue.consume(batch_size=2)
                for msg in batch:
                    start = time.time()
                    app = LoanApplication(msg.payload['app_id'], msg.payload['credit_score'], msg.payload['dti_ratio'], msg.payload['loan_amount'], 12)
                    if rng.random() < 0.10:  # 10% failure
                        self.queue.nack(msg, 'CIC timeout')
                        errors += 1
                    else:
                        worker.process(app)
                        self.queue.ack(msg.msg_id)
                        latencies.append((time.time() - start) * 1000 + rng.gauss(2500, 400))
            if round_n > 50: break  # Safety

        slo = SLOTargets()
        report = compute_slo_report(latencies, errors, self.queue.depth, slo)
        report['total_processed'] = len(latencies)
        report['dlq_size'] = len(self.queue._dlq)
        report['rounds'] = round_n
        return report

# Run
fleet = LoanBotFleetSolution(n_workers=5)
apps = [LoanApplication(f'FC-2025-{i:03d}', 500+rng.randint(0,300), round(0.25+rng.random()*0.25, 2), 200_000_000, 12+rng.randint(0,24)) for i in range(100)]
report = fleet.run_simulation(apps)

print(f'\n=== Fleet Simulation Report ===')
print(f'Total processed: {report["total_processed"]}/100')
print(f'DLQ size: {report["dlq_size"]} (permanent failures)')
print(f'p50: {report["p50_ms"]:.0f}ms | p99: {report["p99_ms"]:.0f}ms {"✅" if report["slo_p99_ok"] else "❌"}')
print(f'Error rate: {report["error_rate_pct"]:.1f}% {"✅" if report["slo_errors_ok"] else "❌"}')